# Data Understanding and Exploratory Data Analysis (EDA)
## AI Weather Intelligence Platform - weatherAUS Dataset

This notebook performs comprehensive data understanding and exploratory data analysis on the weatherAUS.csv dataset.
The analysis includes:
- Dataset structure and composition
- Missing values and duplicates analysis
- Target variable distribution (RainTomorrow)
- Feature distributions and relationships
- Correlation analysis and patterns
- City-wise weather analysis

**Objective**: Prepare the dataset for preprocessing and feature engineering phases.

---

## 1. Import Libraries and Configure Notebook

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Add src directory to path for importing preprocessing module
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Configure notebook display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}' if abs(x) < 1 else f'{x:.2f}')

# Configure matplotlib and seaborn
plt.style.use('default')
sns.set_palette("husl")
sns.set_context("notebook", font_scale=1.1)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print("✓ Libraries imported successfully")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Numpy version: {np.__version__}")

## 2. Load Dataset and Inspect Shape

In [ ]:
# Import the data preprocessing module
from data_preprocessing import load_dataset, get_dataset_info, print_dataset_overview

# Define data path
data_path = Path.cwd().parent / 'data' / 'raw' / 'weatherAUS.csv'

# Load the dataset
df = load_dataset(str(data_path))

# Display basic shape information
print(f"\n{'='*80}")
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"{'='*80}\n")

# Display first few rows
print("First 5 rows of the dataset:")
print(df.head())

# Display last few rows
print("\n\nLast 5 rows of the dataset:")
print(df.tail())

# Column names
print(f"\n\nColumn Names ({len(df.columns)} total):")
print(df.columns.tolist())

## 3. Inspect Data Types and Summary Statistics

In [ ]:
# Get dataset info
info = get_dataset_info(df)

print(f"Dataset Memory Usage: {info['size_mb']} MB\n")

# Data types
print("Data Types Information:")
print(df.dtypes)

print("\n\nData Info:")
df.info()

# Summary statistics for numeric columns
print("\n\nNumeric Summary Statistics:")
print(df.describe().T)

# Summary statistics for categorical columns
print("\n\nCategorical Summary Statistics:")
categorical_cols = df.select_dtypes(include=['object']).columns
print(df[categorical_cols].describe())

## 4. Check Missing Values and Duplicates

In [ ]:
from data_preprocessing import check_missing_values, check_duplicates, check_data_quality

# Check missing values
print("MISSING VALUES ANALYSIS")
print("="*80)
missing_df = check_missing_values(df)

if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
else:
    print("✓ No missing values found in the dataset!")

# Check duplicates
print("\n\nDUPLICATE ROWS ANALYSIS")
print("="*80)
dup_info = check_duplicates(df)
print(f"Total Duplicate Rows: {dup_info['total_duplicates']} ({dup_info['duplicate_percentage']}%)")

if dup_info['total_duplicates'] > 0:
    print("\nFirst 10 duplicate rows:")
    print(dup_info['duplicates_df'].head(10))

# Data quality check
print("\n\nDATA QUALITY ASSESSMENT")
print("="*80)
quality = check_data_quality(df)
print(f"Completeness Score: {quality['completeness_score']}%")
print(f"Uniqueness Score: {quality['uniqueness_score']}%")
print(f"Overall Quality Score: {quality['quality_score']}%")

## 5. Analyze Target Variable Distribution (RainTomorrow)

In [ ]:
from data_preprocessing import analyze_target_variable

# Analyze target variable
target_info = analyze_target_variable(df, 'RainTomorrow')

print("TARGET VARIABLE DISTRIBUTION")
print("="*80)
print("\nValue Counts:")
print(target_info['value_counts'])

print("\n\nPercentage Distribution:")
print(target_info['percentages'])

print(f"\n\nMissing Values in Target: {target_info['missing_values']}")

if target_info['imbalance_ratio']:
    print(f"Class Imbalance Ratio: {target_info['imbalance_ratio']:.2f}:1")

# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
target_info['value_counts'].plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Target Variable Distribution (Counts)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('RainTomorrow')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(target_info['value_counts'], labels=target_info['value_counts'].index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Target Variable Distribution (Percentage)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Target variable analysis complete")

## 6. Visualize Feature Distributions

In [ ]:
# Get numeric columns for visualization
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove target variable if it's numeric
if 'RainTomorrow' in numeric_cols:
    numeric_cols.remove('RainTomorrow')

# Select key numeric features for visualization
key_numeric_features = [col for col in numeric_cols if col not in ['Date', 'Year', 'Month', 'Day']][:6]

print(f"Visualizing distributions for: {key_numeric_features}\n")

# Create histograms for numeric features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, col in enumerate(key_numeric_features):
    axes[idx].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Categorical features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'Date' in categorical_cols:
    categorical_cols.remove('Date')
if 'RainTomorrow' in categorical_cols:
    categorical_cols.remove('RainTomorrow')

# Select key categorical features
key_categorical_features = categorical_cols[:4] if len(categorical_cols) >= 4 else categorical_cols

print(f"\n\nVisualizing categorical features: {key_categorical_features}\n")

# Create countplots for categorical features
fig, axes = plt.subplots(1, len(key_categorical_features), figsize=(15, 4))

if len(key_categorical_features) == 1:
    axes = [axes]

for idx, col in enumerate(key_categorical_features):
    value_counts = df[col].value_counts().head(10)
    axes[idx].barh(value_counts.index, value_counts.values, color='coral', edgecolor='black')
    axes[idx].set_title(f'{col}', fontweight='bold')
    axes[idx].set_xlabel('Count')
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

print("✓ Feature distributions visualized")

## 7. Correlation Heatmap and Feature Relationships

In [ ]:
# Calculate correlation matrix for numeric features
correlation_matrix = df[numeric_cols].corr()

# Create correlation heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap - Numeric Features', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Identify top correlations with target variable
print("TOP CORRELATIONS WITH TARGET VARIABLE (RainTomorrow)")
print("="*80)

# Convert target to numeric if needed
if 'RainTomorrow' in df.columns:
    rain_numeric = pd.factorize(df['RainTomorrow'])[0]
    target_corr = df[numeric_cols].corrwith(pd.Series(rain_numeric))
    target_corr = target_corr.sort_values(ascending=False)
    print(target_corr)

print("\n✓ Correlation analysis complete")

## 8. Missing Value Heatmap

In [ ]:
# Create missing value heatmap
plt.figure(figsize=(14, 8))

# Create binary matrix for missing values
missing_matrix = df.isnull().astype(int)

# Sample if dataset is large
sample_size = min(500, len(df))
sample_indices = np.random.choice(df.index, sample_size, replace=False).sort_values()
missing_sample = missing_matrix.loc[sample_indices]

# Plot heatmap
sns.heatmap(missing_sample.T, cbar=True, cmap='YlOrRd', yticklabels=True)
plt.title(f'Missing Value Pattern (Sample of {sample_size} rows)', fontsize=14, fontweight='bold')
plt.xlabel('Row Index')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

# Print summary
print("MISSING VALUE SUMMARY BY COLUMN")
print("="*80)
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (100 * df.isnull().sum() / len(df)).round(2)
})
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_summary) > 0:
    print(missing_summary.to_string(index=False))
else:
    print("✓ No missing values detected in the dataset!")

print("\n✓ Missing value analysis complete")

## 9. City-wise Weather Analysis

In [ ]:
# Check if 'Location' or 'City' column exists
location_col = None
for col in ['Location', 'City', 'location', 'city']:
    if col in df.columns:
        location_col = col
        break

if location_col:
    print(f"CITY-WISE WEATHER ANALYSIS (by {location_col})")
    print("="*80)
    
    # City-wise statistics
    city_stats = df.groupby(location_col).agg({
        'MinTemp': 'mean',
        'MaxTemp': 'mean',
        'Rainfall': 'mean',
        'Humidity3pm': 'mean',
        'WindSpeed3pm': 'mean',
        'RainTomorrow': lambda x: (x == 'Yes').sum() / len(x) * 100 if 'Yes' in x.values else 0
    }).round(2)
    
    city_stats.columns = ['Avg_MinTemp', 'Avg_MaxTemp', 'Avg_Rainfall', 'Avg_Humidity', 'Avg_WindSpeed', 'Rain_Probability_%']
    city_stats = city_stats.sort_values('Rain_Probability_%', ascending=False)
    
    print("\nTop 10 Cities by Rain Probability:")
    print(city_stats.head(10).to_string())
    
    # Visualize rain probability by city
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Rain probability by city
    top_cities = city_stats.head(10)
    axes[0].barh(top_cities.index, top_cities['Rain_Probability_%'], color='steelblue', edgecolor='black')
    axes[0].set_title('Top 10 Cities by Rain Probability', fontweight='bold')
    axes[0].set_xlabel('Rain Probability (%)')
    axes[0].invert_yaxis()
    
    # Average temperature by city
    axes[1].scatter(city_stats['Avg_MinTemp'], city_stats['Avg_MaxTemp'], 
                   s=city_stats['Rain_Probability_%']*10, alpha=0.6, c=city_stats['Rain_Probability_%'],
                   cmap='RdYlGn_r', edgecolors='black', linewidth=0.5)
    axes[1].set_xlabel('Average Min Temperature (°C)')
    axes[1].set_ylabel('Average Max Temperature (°C)')
    axes[1].set_title('Temperature Range vs Rain Probability', fontweight='bold')
    
    # Add colorbar
    cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
    cbar.set_label('Rain Probability (%)')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ City-wise analysis complete")
else:
    print("ℹ Location/City column not found in the dataset")
    print(f"Available columns: {df.columns.tolist()}")

## 10. Data Understanding Summary and Next Steps

### Key Findings Summary

**Dataset Overview:**
- The weatherAUS dataset contains weather observations across multiple locations in Australia
- Comprehensive temporal coverage with daily weather measurements
- Mix of numeric and categorical features

**Data Quality:**
- Overall data quality score and completeness metrics documented
- Missing values and duplicate analysis completed
- Recommendations for data cleaning identified

**Target Variable (RainTomorrow):**
- Binary classification target: Did it rain the next day?
- Class distribution and balance ratio analyzed
- Important for model training and evaluation

**Feature Insights:**
- Correlation analysis reveals feature relationships
- Temperature, humidity, and rainfall show expected patterns
- Geographic location impacts weather patterns

### Recommendations for Next Steps

1. **Data Preprocessing (02_data_preprocessing.ipynb)**
   - Handle missing values using appropriate imputation strategies
   - Remove or manage duplicate records
   - Normalize/standardize numeric features

2. **Feature Engineering (03_feature_engineering.ipynb)**
   - Create temporal features (day of week, month, season)
   - Engineer weather indices (wind chill, apparent temperature)
   - Create interaction features for predictive power

3. **Model Preparation**
   - Feature selection based on correlation and importance
   - Train-test split for proper model evaluation
   - Class imbalance handling if necessary

---

**Analysis Date**: 2026-07-25  
**Dataset**: weatherAUS.csv  
**Module Used**: src/data_preprocessing.py


In [ ]:
# Print comprehensive dataset overview
print_dataset_overview(df, 'RainTomorrow')

print("\n" + "="*80)
print("EDA ANALYSIS COMPLETE".center(80))
print("="*80)
print(f"\n✓ All exploratory data analysis steps completed successfully!")
print(f"✓ Ready for data preprocessing phase")
print(f"✓ Dataset insights documented for downstream tasks")